In [1]:
import datetime
import gc
import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [2]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

In [3]:
replica_paths = sorted(glob.glob(f"./replicas/*.keras"))

# load the models
replicas = [tf.keras.models.load_model(path, compile = False, safe_mode = False) for path in replica_paths]

number_of_replicas = len(replicas)

print(f"Loaded {number_of_replicas} replica models.")

TypeError: Could not deserialize class 'Functional' because its parent module tf_keras.src.engine.functional cannot be imported. Full object config: {'module': 'tf_keras.src.engine.functional', 'class_name': 'Functional', 'config': {'name': 'model', 'trainable': True, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_input_shape': [None, 4], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_values'}, 'registered_name': None, 'name': 'input_values', 'inbound_nodes': []}, {'module': 'keras.layers', 'class_name': 'Dense', 'config': {'name': 'dense', 'trainable': True, 'dtype': 'float32', 'units': 48, 'activation': 'tanh', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotNormal', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': N

In [7]:
individual_replica_predictions = sorted(glob.glob(f"./data/replica_*_test_predictions.csv"))
replica_dfs = [pd.read_csv(f) for f in individual_replica_predictions]

list_of_dfs = []

for replica_index, filename in enumerate(individual_replica_predictions):
    df = pd.read_csv(filename)
    df['replica_id'] = replica_index
    list_of_dfs.append(df)

# Combine into one large master dataframe
master_df = pd.concat(list_of_dfs, ignore_index = True)

In [8]:
reference = replica_dfs[0][['t', 'x_b', 'q_squared', 'phi']]

for i, df in enumerate(replica_dfs[1:], start = 1):

    current = df[['t', 'x_b', 'q_squared', 'phi']]

    if not reference.equals(current):
        raise ValueError(f"Replica {i} is not aligned!")

In [ ]:
predictions = np.stack([
    df[['pred_xsec', 'pred_bsa']].values
    for df in replica_dfs
])

average_prediction = predictions.mean(axis = 0)
standard_dev_prediction  = predictions.std(axis = 0)

In [ ]:
# this is trento convention: -pi to pi:
phi_smooth = np.linspace(-np.pi, np.pi, 361) 

for (t_val, xb_val, q2_val), group in grouped:
    print(f"Processing t = {t_val}, xb = {xb_val}, Q2 = {q2_val}")

    group = group.sort_values('phi')

    xsec_err = 0.
    bsa_err = 0.

    indices = group.index.values

    xsec_pred = average_prediction[indices, 0]
    bsa_pred = average_prediction[indices, 1]
    xsec_std = standard_dev_prediction[indices, 0]
    bsa_std = standard_dev_prediction[indices, 1]

    x_smooth = np.column_stack([
        np.full_like(phi_smooth, t_val),
        np.full_like(phi_smooth, xb_val),
        np.full_like(phi_smooth, q2_val),
        phi_smooth
    ])

    smooth_preds_all = np.array([
        model.predict(x_smooth) for model in replicas
    ])

    smooth_mean = np.mean(smooth_preds_all, axis = 0)
    smooth_std  = np.std(smooth_preds_all, axis = 0)

    xsec_smooth_mean = smooth_mean[:, 0]
    bsa_smooth_mean  = smooth_mean[:, 1]

    xsec_smooth_std = smooth_std[:, 0]
    bsa_smooth_std  = smooth_std[:, 1]

    phi = group['phi'].values
    xsec_actual = group['unp_beam_unp_target_xsec'].values
    bsa_actual = group['unp_target_bsa'].values

    xsec_res = xsec_actual - xsec_pred
    bsa_res = bsa_actual - bsa_pred

    chi2_xsec = np.sum(xsec_res**2) / len(phi)
    chi2_bsa = np.sum(bsa_res**2) / len(phi)

    residuals_figure, axes = plt.subplots(2, 2, figsize = (10, 8), sharex = 'col')

    axes[0, 0].plot(phi_smooth, xsec_smooth_mean, color = 'red', lw = 2, label = rf'Replica Average ($N = {number_of_replicas}$)')
    axes[0, 0].fill_between(
        phi_smooth,
        xsec_smooth_mean - xsec_smooth_std,
        xsec_smooth_mean + xsec_smooth_std,
        color = 'red',
        alpha = 0.3,
        label = r'$\sigma$ band'
    )
    axes[0, 0].errorbar(
        phi, xsec_actual, yerr = xsec_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    axes[0, 0].set_ylabel(r"$d^{4}\sigma$", fontsize=14)
    axes[0, 0].set_title(rf"Cross Section ($\chi^2_\nu = {chi2_xsec:.3f}$)")
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle=':', alpha=0.6)

    axes[0, 1].scatter(phi, xsec_res, color='blue', alpha=0.6)
    axes[0, 1].axhline(0, color='black', linestyle='--')
    axes[0, 1].set_title("Residuals")
    axes[0, 1].grid(True, linestyle=':', alpha=0.6)

    axes[1, 0].plot(phi_smooth, bsa_smooth_mean, color='green', lw=2, label=rf'Replica Average ($N = {number_of_replicas}$)')
    axes[1, 0].fill_between(
        phi_smooth,
        bsa_smooth_mean - bsa_smooth_std,
        bsa_smooth_mean + bsa_smooth_std,
        color='green',
        alpha=0.3,
        label=r'$\sigma$ band'
    )
    axes[1, 0].errorbar(
        phi, bsa_actual, yerr = bsa_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    
    axes[1, 0].set_ylabel("BSA", fontsize=14)
    axes[1, 0].set_xlabel(r"$\phi$ (radians)", fontsize=14)
    axes[1, 0].set_title(rf"BSA ($\chi^2_\nu = {chi2_bsa:.3f}$)")
    axes[1, 0].legend()
    axes[1, 0].grid(True, linestyle=':', alpha=0.6)

    axes[1, 1].scatter(phi, bsa_res, color='purple', alpha=0.6)
    axes[1, 1].axhline(0, color='black', linestyle='--')
    axes[1, 1].set_xlabel(r"$\phi$ (radians)", fontsize=14)
    axes[1, 1].set_title("Residuals")
    axes[1, 1].grid(True, linestyle=':', alpha=0.6)
    
    residuals_figure.suptitle(
        rf"$t = {t_val:.3f}$, $x_\textrm{{B}} = {xb_val:.3f}$, $Q^2={q2_val:.3f}$",
        fontsize=16
    )

    filename = f"./plots/t{t_val:.3f}_xb{xb_val:.3f}_q2{q2_val:.3f}_residuals_v1"
    for extension in ['png', 'eps']:
        residuals_figure.savefig(
            fname = f"{filename}.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(residuals_figure)

Processing t = -0.45, xb = 0.215, Q2 = 1.88


NameError: name 'replicas' is not defined